# Module 5: PySpark SQL

Run in Google Colab or a local PySpark environment.

In [ ]:
# Run this cell first
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder.appName("LearningModules").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Load datasets (adjust path if needed)
students_df = spark.read.csv("../data/students.csv", header=True, inferSchema=True)
enrollments_df = spark.read.csv("../data/enrollments.csv", header=True, inferSchema=True)

## Registering Temp Views

To use SQL syntax on DataFrames, first register them as temporary views.

In [ ]:
# Register DataFrames as SQL views
students_df.createOrReplaceTempView("students")
enrollments_df.createOrReplaceTempView("enrollments")

## SELECT and WHERE

In [ ]:
# Basic SELECT
spark.sql("SELECT name, gpa FROM students").show(5)

# WHERE clause
spark.sql("SELECT * FROM students WHERE gpa > 3.5").show()

## GROUP BY

In [ ]:
# Count students per GPA bracket
spark.sql("""
    SELECT 
        CASE 
            WHEN gpa >= 3.5 THEN 'High'
            WHEN gpa >= 3.0 THEN 'Medium'
            ELSE 'Low'
        END AS gpa_bracket,
        COUNT(*) as student_count
    FROM students
    GROUP BY gpa_bracket
""").show()

## JOIN Queries

In [ ]:
# Inner join students with enrollments
spark.sql("""
    SELECT s.name, s.gpa, e.course_id
    FROM students s
    JOIN enrollments e ON s.student_id = e.student_id
""").show(5)

## Subqueries

In [ ]:
# Subquery: students above average GPA
spark.sql("""
    SELECT name, gpa
    FROM students
    WHERE gpa > (SELECT AVG(gpa) FROM students)
    ORDER BY gpa DESC
""").show()

## Common Table Expressions (CTEs)

In [ ]:
# CTE to find students with the most enrollments
spark.sql("""
    WITH enrollment_counts AS (
        SELECT student_id, COUNT(*) AS num_courses
        FROM enrollments
        GROUP BY student_id
    )
    SELECT s.name, s.gpa, ec.num_courses
    FROM students s
    JOIN enrollment_counts ec ON s.student_id = ec.student_id
    ORDER BY ec.num_courses DESC
""").show(5)

## Practice Problem 1: SELECT with Conditions

Write a SQL query to find all students with GPA between 3.0 and 3.8 (inclusive), ordered by name.

<details><summary>Hint</summary>Use BETWEEN in the WHERE clause and ORDER BY name.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
spark.sql("""
    SELECT * FROM students
    WHERE gpa BETWEEN 3.0 AND 3.8
    ORDER BY name
""").show()

## Practice Problem 2: GROUP BY with HAVING

Write a SQL query to find course_ids that have more than 2 enrollments.

<details><summary>Hint</summary>GROUP BY course_id, use HAVING COUNT(*) > 2.</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
spark.sql("""
    SELECT course_id, COUNT(*) as enrollment_count
    FROM enrollments
    GROUP BY course_id
    HAVING COUNT(*) > 2
    ORDER BY enrollment_count DESC
""").show()

## Practice Problem 3: CTE with JOIN

Using a CTE, find the average GPA of students enrolled in each course.

<details><summary>Hint</summary>Join students and enrollments in a CTE, then GROUP BY course_id and compute AVG(gpa).</details>

In [ ]:
# Your code here


<details><summary>Solution</summary></details>

In [ ]:
# Solution
spark.sql("""
    WITH student_courses AS (
        SELECT s.student_id, s.name, s.gpa, e.course_id
        FROM students s
        JOIN enrollments e ON s.student_id = e.student_id
    )
    SELECT course_id, ROUND(AVG(gpa), 2) AS avg_gpa
    FROM student_courses
    GROUP BY course_id
    ORDER BY avg_gpa DESC
""").show()